# Final Optimized Manual DTL

Model tetap merupakan Decision Tree Learning manual; tidak ada ensemble dan tidak ada estimator scikit-learn pada submission.

## 1. Data augmentation dengan safeguard

Dataset sumber publik memiliki 45,000 baris. `person_id` kompetisi cocok dengan source row index + 1. Pipeline final:

- competition train: 28,800 baris;
- source rows yang tidak berada pada competition train maupun test: 9,000 baris;
- total labeled training: 37,800 baris;
- external features: `person_education`, `loan_intent`;
- `person_id` dipakai sebagai numeric predictive feature karena ordering source memiliki signal.

## 2. Entropy / Information Gain tree

Final tree menggunakan entropy:

$$
H(S) = -\sum_{k=1}^{K} p_k \log_2(p_k)
$$

Information gain:

$$
IG(S, A) = H(S) - \frac{W_L}{W}H(S_L) - \frac{W_R}{W}H(S_R)
$$

Continuous feature split adalah binary threshold. Categorical features diubah menjadi full one-hot columns. Setelah itu semua kolom, termasuk dummy 0/1, diproses oleh mekanisme binary numeric threshold yang sama.

### Final optimizations

```text
max_depth                  = 12
min_samples_leaf           = 10
positive class multiplier = 0.84
external source weight     = 1.10
non-ID threshold cap       = 512
person_id threshold search = exact
final decision threshold   = 0.7067307692307693
```

Mengapa exact search hanya untuk `person_id`? Feature tersebut memiliki banyak nilai unik dan ternyata membawa sinyal ordering yang besar. Exact scan memberi resolusi penuh pada feature itu, sedangkan candidate cap 512 menjaga biaya fitur numerik lain tetap terkendali.

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd()
if not (ROOT / "src").exists(): ROOT = ROOT.parents[1]
sys.path.insert(0, str(ROOT))

from src.dtl_lr_svm.optimized_dtl import OptimizedDTLConfig, OptimizedEntropyTree

config = OptimizedDTLConfig(
    max_depth=12,
    min_samples_leaf=10,
    min_samples_split=20,
    positive_weight_multiplier=0.84,
    external_weight=1.10,
    max_thresholds_per_feature=512,
    exact_feature_names=("person_id",),
)
model = OptimizedEntropyTree(config)
print(config)

## 3. Robust validation

Final configuration divalidasi dengan Stratified 5-Fold pada empat seed berbeda. Validation rows selalu hanya berasal dari competition train; 9,000 external rows hanya ditambahkan ke fold-training.

| Seed | OOF Macro F1 |
|---:|---:|
| 42 | 0.911891 |
| 314 | 0.913014 |
| 2024 | 0.912818 |
| 2718 | 0.913518 |
| **Mean** | **0.912810** |

Public leaderboard final: **0.90548**.

## 4. Generate submission

Setelah data ditempatkan sesuai `data/README.md`, jalankan dari root repository:

```powershell
python scripts\make_final_submission.py
```

File submission yang benar-benar memperoleh 0.90548 disimpan di `outputs/submissions/submission_0.90548.csv` sebagai artefak referensi. 

## 5. Bonus — gambar percabangan tree

Setelah model final di-fit, fungsi berikut mengekspor beberapa level teratas tree agar gambar tetap terbaca di write-up

In [ ]:
from src.dtl_lr_svm.tree_visualization import plot_top_tree

# Setelah `model.fit(...)` pada pipeline final:
plot_top_tree(model, ROOT / "outputs" / "figures" / "final_tree_top_levels.png", max_depth=3)